## Notes for running
- If you don't have the CSVs in the notebook directory, update the `PERF_PATH` and `LOOKUP_PATH` variables in the next cell.
- Normality tests behave differently with large samples: Shapiro is unreliable for n > 5000 (we'll sample in that case). KS against a standard normal is a useful check on standardized data.
- Install dependencies if needed (next code cell).

In [ ]:
# Install dependencies if not already available (uncomment and run if needed).
# import sys
# !{sys.executable} -m pip install pandas numpy scipy matplotlib seaborn statsmodels scikit-learn

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
from statsmodels.graphics.gofplots import qqplot
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression, f_classif
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (8,5)

In [ ]:
# Paths - update if your files are elsewhere
PERF_PATHS = [Path.cwd() / 'university_performance.csv', Path.cwd() / 'data' / 'university_performance.csv', Path.cwd().parent / 'university_performance.csv']
LOOKUP_PATHS = [Path.cwd() / 'university_support_lookup.csv', Path.cwd() / 'data' / 'university_support_lookup.csv', Path.cwd().parent / 'university_support_lookup.csv']
perf = None
lookup = None
for p in PERF_PATHS:
    if p.exists():
        perf = pd.read_csv(p)
        print('Loaded university_performance from', p)
        break
for p in LOOKUP_PATHS:
    if p.exists():
        lookup = pd.read_csv(p)
        print('Loaded university_support_lookup from', p)
        break
if perf is None or lookup is None:
    print('Could not auto-locate one or both CSVs.
Please set `PERF_PATHS` and `LOOKUP_PATHS` to the correct locations and re-run this cell.')

In [ ]:
# Quick preview (only run if files were loaded)
if perf is not None:
    display(perf.head())
    print('
Data shape:', perf.shape)
    display(perf.info())
if lookup is not None:
    display(lookup.head())
    print('
Lookup shape:', lookup.shape)

## Numeric features: skewness & kurtosis

In [ ]:
# Compute skew and kurtosis for numeric columns
if perf is not None:
    num_cols = perf.select_dtypes(include='number').columns.tolist()
    desc = perf[num_cols].agg(['count','mean','std','min','max']).T
    skew_kurt = pd.DataFrame({
        'skew': perf[num_cols].skew(),
        'kurtosis_excess': perf[num_cols].kurtosis()
        })
    stats_df = pd.concat([desc, skew_kurt], axis=1)
    display(stats_df.sort_values(by='skew', key=abs, ascending=False).head(20))
else:
    print('Load datasets first.')

## Normality checks: Shapiro-Wilk and Kolmogorov-Smirnov (KS)

In [ ]:
# Function to run Shapiro and KS (against normal) with safe sampling for large n
def normality_tests(series, max_shapiro_n=5000, sample_n=5000):
    x = series.dropna()
    n = len(x)
    results = {}
    # Shapiro - sample if too big
    if n == 0:
        return {'n':0}    
    if n > max_shapiro_n:
        xs = x.sample(sample_n, random_state=0)
        s_stat, s_p = stats.shapiro(xs)
        results['shapiro_sampled'] = sample_n
    else:
        s_stat, s_p = stats.shapiro(x)
    results['shapiro_stat'] = float(s_stat)
    results['shapiro_p'] = float(s_p)
    # KS test against standard normal on standardized data
    z = (x - x.mean())/x.std(ddof=0)
    ks_stat, ks_p = stats.kstest(z, 'norm')
    results['ks_stat'] = float(ks_stat)
    results['ks_p'] = float(ks_p)
    results['n'] = n
    return results

# Run tests for numeric columns and summarize
if perf is not None:
    norm_tests = {}
    for c in num_cols:
        try:
            norm_tests[c] = normality_tests(perf[c])
        except Exception as e:
            norm_tests[c] = {'error': str(e)}
    norm_df = pd.DataFrame.from_dict(norm_tests, orient='index')
    display(norm_df.sort_values('shapiro_p').head(20))
else:
    print('Load datasets first.')

## QQ plots (visual check)

In [ ]:
# Choose top skewed columns for QQ plotting
if perf is not None:
    top_skew = stats_df['skew'].abs().sort_values(ascending=False).head(6).index.tolist()
    for col in top_skew:
        plt.figure()
        qqplot(perf[col].dropna(), line='s')
        plt.title(f'QQ plot: {col}')
        plt.show()
else:
    print('Load datasets first.')

## Feature-selection workflow (for learners): correlation, variance, mutual information / ANOVA

In [ ]:
# 1) Correlation heatmap and high-correlation removal suggestion
if perf is not None:
    corr = perf[num_cols].corr().abs()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    plt.figure(figsize=(10,8))
    sns.heatmap(corr, mask=mask, cmap='coolwarm', annot=False)
    plt.title('Absolute correlation (numeric features)')
    plt.show()
    # Suggest pairs with high correlation
    high_corr = []
    thresh = 0.9
    for i in range(len(corr.columns)):
        for j in range(i):
            if corr.iloc[i, j] > thresh:
                high_corr.append((corr.index[i], corr.columns[j], corr.iloc[i,j]))
    print('Highly correlated pairs (>|0.9|):')
    if len(high_corr)==0:
        print('None')
    else:
        for r in high_corr:
            print(r)
else:
    print('Load datasets first.')

In [ ]:
# 2) Supervised feature scoring example (auto-detect target)
# Set TARGET_COL to your target variable name if known; otherwise the cell will try to infer one.
TARGET_COL = None  # <-- set this to your target column name if you have one

if perf is not None:
    if TARGET_COL is None:
        # heuristic: look for columns with names suggesting performance/score/grade
        candidates = [c for c in perf.columns if any(k in c.lower() for k in ('score','perf','grade','pass','rate'))]
        if candidates:
            TARGET_COL = candidates[0]
            print('Inferred target column:', TARGET_COL)
    if TARGET_COL is None or TARGET_COL not in perf.columns:
        print('No target found. Set `TARGET_COL` to run supervised feature scoring. Skipping supervised scoring.')
    else:
        X = perf.drop(columns=[TARGET_COL]).select_dtypes(include='number').fillna(0)
        y = perf[TARGET_COL]
        # Decide regression vs classification by unique count heuristic
        if pd.api.types.is_numeric_dtype(y) and y.nunique() > 10:
            print('Detected regression target (numeric). Using mutual_info_regression-like approach.')
            from sklearn.feature_selection import mutual_info_regression
            mi = mutual_info_regression(X, y, random_state=0)
            score_df = pd.Series(mi, index=X.columns).sort_values(ascending=False)
        else:
            print('Detected classification or categorical target. Using mutual_info_classif / ANOVA F-test.')
            # try mutual info classification
            try:
                mi = mutual_info_classif(X, y.fillna(0), random_state=0)
                score_df = pd.Series(mi, index=X.columns).sort_values(ascending=False)
            except Exception:
                fvals, pvals = f_classif(X, y.fillna(0))
                score_df = pd.Series(fvals, index=X.columns).sort_values(ascending=False)
        display(score_df.head(30))
else:
    print('Load datasets first.')

## Transform skewed features and re-evaluate

In [ ]:
# Apply log1p to highly skewed features and show before/after Shapiro p-values for a quick check
if perf is not None:
    skewed = stats_df['skew'].abs().sort_values(ascending=False).head(6).index.tolist()
    transform_results = {}
    for col in skewed:
        orig = perf[col].dropna()
        # avoid negative values for log1p: shift if necessary
        shift = 0
        if (orig <= -1).any():
            shift = (orig.min() * -1) + 0.001
        transformed = np.log1p(orig + shift)
        orig_test = normality_tests(orig)
        trans_test = normality_tests(transformed)
        transform_results[col] = {
            'orig_shapiro_p': orig_test.get('shapiro_p'),
            'trans_shapiro_p': trans_test.get('shapiro_p'),
            'orig_skew': float(orig.skew()),
            'trans_skew': float(pd.Series(transformed).skew())
        }
    display(pd.DataFrame(transform_results).T)
else:
    print('Load datasets first.')

## Summary & guidance for learners
- **Skew & Kurtosis:** Used to spot strongly asymmetric or heavy-tailed features. Large skew often suggests transformation (log1p, Box-Cox) before parametric methods.
- **Shapiro-Wilk:** Good for small-to-moderate sample sizes (n <= 5000). If p < 0.05, data deviates from normality (reject H0).
- **KS test (against normal):** Works on standardized data; complements Shapiro but can be sensitive to tails. Use together for a robust view.
- **QQ plots:** Visual check — systematic deviation from the line signals non-normality.
- **Feature selection impact:** Many classical tests/lines (t-test, ANOVA, linear models) assume approximate normality of residuals or features; use normality checks and transformations to decide whether to transform features or use non-parametric/robust feature scorers (mutual information, tree-based methods).
- **Next steps:** Set the `TARGET_COL` and rerun supervised feature scoring; try tree-based feature importances or permutation importance for model-agnostic ranking.